In [6]:
import sys
from pathlib import Path

# project root = parent of /analysis
project_root = Path().resolve().parent

# fix sys.path (for imports)
sys.path.append(str(project_root))

# Analysing the created jsonl file

In [7]:
import json
from pathlib import Path
from pprint import pprint
import statistics
from src.config import DEFAULT_OUTPUT_JSONL

jsonl_path = project_root / DEFAULT_OUTPUT_JSONL


def iter_entries(path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            yield json.loads(line)


entries = list(iter_entries(jsonl_path))

print(f"Loaded {len(entries)} project entries")


# Show first project
project = entries[0]

print("\n=== PROJECT METADATA ===")
pprint(project["project"])


print("\n=== SOURCE FILE COUNT ===")
print(len(project["sources"]))


# -----------------------------
# NEW STATS COMPUTATION
# -----------------------------

file_function_counts = [
    len(source["functions"])
    for source in project["sources"]
]

total_functions = sum(file_function_counts)

avg_functions = statistics.mean(file_function_counts) if file_function_counts else 0
min_functions = min(file_function_counts) if file_function_counts else 0
max_functions = max(file_function_counts) if file_function_counts else 0


print("\n=== FUNCTION STATISTICS ===")
print(f"Total functions in project: {total_functions}")
print(f"Avg functions per file: {avg_functions:.2f}")
print(f"Min functions per file: {min_functions}")
print(f"Max functions per file: {max_functions}")


# Show first few source files
for source in project["sources"][:3]:

    print("\n" + "=" * 80)
    print("FILE:", source["relative_path"])

    print("Functions:", len(source["functions"]))

    for fn in source["functions"][:2]:

        print("\n  FUNCTION:", fn["qualified_name"])
        print("  Signature:", fn["signature"])
        print("  Lines:", f'{fn["start_line"]}-{fn["end_line"]}')

        if fn.get("metrics"):
            print("  LOC:", fn["metrics"]["loc"])
            print("  Tokens:", fn["metrics"]["token_count"])

        if fn.get("decorators"):
            print("  Decorators:", fn["decorators"])

        if fn.get("modifiers"):
            print("  Modifiers:", fn["modifiers"])

        if fn.get("code"):
            print("\n  RAW CODE PREVIEW:")
            print("-" * 40)

            preview = fn["code"]["raw"]
            print(preview)

            if len(fn["code"]["raw"]) > 500:
                print("...")

Loaded 1 project entries

=== PROJECT METADATA ===
{'language': 'python', 'name': 'calc-test', 'repository_url': ''}

=== SOURCE FILE COUNT ===
5

=== FUNCTION STATISTICS ===
Total functions in project: 16
Avg functions per file: 3.20
Min functions per file: 2
Max functions per file: 4

FILE: main.py
Functions: 2

  FUNCTION: run_calculator
  Signature: run_calculator
  Lines: 28-111
  LOC: 84
  Tokens: 251

  RAW CODE PREVIEW:
----------------------------------------
def run_calculator():

    while True:

        print_menu()
        choice = input("Choose an operation: ")

        result = None
        expression = None

        if not validate_choice(choice):
            print("Invalid option\n")
            continue

        # -------------------------
        # BASIC OPERATIONS (1–4)
        # -------------------------
        if choice in ["1", "2", "3", "4"]:

            a = get_number("Enter first number: ")
            b = get_number("Enter second number: ")

            if 

# Analyzing the pairs file

In [9]:

import pandas as pd 
from src.config import DEFAULT_PAIRS_CSV
path = project_root / DEFAULT_PAIRS_CSV
df = pd.read_csv(path)
 

print("\nMissing values per column:")
print(df.isnull().sum())

df["same_entry"] = df["entry_a_id"] == df["entry_b_id"]

print("\nPairs from same entry:", df["same_entry"].sum())
print("Pairs from different entries:", (~df["same_entry"]).sum())

df["len_a"] = df["code_a"].astype(str).apply(len)
df["len_b"] = df["code_b"].astype(str).apply(len)

print("\nCode length statistics (A):")
print(df["len_a"].describe())

print("\nCode length statistics (B):")
print(df["len_b"].describe())

df["pair_key"] = df.apply(
    lambda r: tuple(sorted([r["function_a_id"], r["function_b_id"]])),
    axis=1
)

dup_pairs = df.duplicated("pair_key").sum()
print("\nDuplicate function pairs (unordered):", dup_pairs)

print("\nTop entries by number of pairs:")
print(df["entry_a_id"].value_counts().head(10))

print("\nExample pair:")
row = df.sample(1).iloc[0]

print("\nA ID:", row["function_a_id"])
print("B ID:", row["function_b_id"])
print("\nA code:\n", row["code_a"][:500])
print("\nB code:\n", row["code_b"][:500])


Missing values per column:
function_a_id    0
entry_a_id       0
code_a           0
function_b_id    0
entry_b_id       0
code_b           0
dtype: int64

Pairs from same entry: 120
Pairs from different entries: 0

Code length statistics (A):
count     120.000000
mean      434.916667
std       774.065417
min        32.000000
25%        37.000000
50%       181.000000
75%       312.250000
max      2455.000000
Name: len_a, dtype: float64

Code length statistics (B):
count    120.000000
mean     165.833333
std      111.588254
min       32.000000
25%       51.000000
50%      158.000000
75%      253.000000
max      357.000000
Name: len_b, dtype: float64

Duplicate function pairs (unordered): 0

Top entries by number of pairs:
entry_a_id
calc-test    120
Name: count, dtype: int64

Example pair:

A ID: c88a426c-b543-5122-96ec-5e6be17986b6
B ID: 465c79aa-e5be-55c2-b9f6-efbbe944983c

A code:
 def get_number(message):\n\n    while True:\n\n        try:\n            return float(input(message))\n